In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))
from langchain.agents import create_agent
from langchain.messages import HumanMessage, ToolMessage, AIMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from agentic.tools import NOTEBOT_TOOLS
from dotenv import load_dotenv
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from langchain.agents.middleware import ToolRetryMiddleware, ToolCallLimitMiddleware
from langchain.agents.middleware import (wrap_tool_call,before_model,after_model,AgentState)
from collections.abc import Callable
from langchain.tools.tool_node import ToolCallRequest
from langgraph.runtime import Runtime
from typing import Any
from langgraph.stream import StreamTransformer, StreamChannel
import pprint
pprint = pprint.pprint

In [ ]:
load_dotenv()


In [ ]:
print("LANGSMITH_API_KEY:", os.getenv("LANGSMITH_API_KEY"))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))

In [ ]:
model = ChatOllama(
    model="qwen3.5:4b-mlx",
    reasoning=False,
    temperature=0.0,
    top_p=0.5
)

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

checkpointer = SqliteSaver(sqlite3.connect("../db/checkpoint.db",check_same_thread=False))

In [ ]:
import sqlite3
from langgraph.store.sqlite import SqliteStore

conn = sqlite3.connect("../db/store.db",autocommit=True,check_same_thread=False)
store = SqliteStore(conn=conn)

In [ ]:
def chat_with_agent(agent,t_id):
    
    config = {"configurable": {"thread_id": t_id}}

    class MyCustomTransformer(StreamTransformer):
        required_stream_modes = ("custom",)

        def __init__(self, scope: tuple[str, ...] = ()) -> None:
            super().__init__(scope)
            # 1. Define the channel
            self.log = StreamChannel()

        def init(self) -> dict:
            # 2. Key "my_custom" will be the name used in interleave()
            return {"my_custom": self.log}

        def process(self, event) -> bool:
            if event["method"] == "custom":
                self.log.push(event["params"]["data"])
            return True
    msg = input("Enter: ")

    while msg:
        stream = agent.stream_events(
            {"messages": [HumanMessage(msg)]},
            config=config,
            transformers=[MyCustomTransformer],
            version="v3",
        )
        for name, item in stream.interleave("my_custom", "messages"):
            if name == "my_custom":
                print(f"Tool update: {item}")
            elif name == "messages":
                for j in item.text:
                    print(j,end="",flush=True)
        print()
        while stream.interrupted:
            interrupt = stream.interrupts[-1].value

            request = interrupt["action_requests"][-1]
            review = interrupt["review_configs"][-1]

            print("\nAction:")
            print(request["description"])

            print("\nAllowed decisions:")
            for d in review["allowed_decisions"]:
                print("-", d)

            choice = input("\nYour choice: ").strip()

            if choice == "approve":
                decision = {
                    "type": "approve"
                }

            elif choice == "reject":
                feedback = input("Reason (optional): ")
                if feedback:
                    feedback = f"Unsuccessful. The user has rejected the tool call with the following Feedback:{feedback}. Try Again"

                decision = {
                    "type": "reject",
                    "message": feedback
                }

            elif choice == "respond":
                reply = input("Response to tool: ")
                decision = {
                    "type": "respond",
                    "message": reply
                }

            elif choice == "edit":
                print(f"\nTool: {request['name']}")

                new_args = request["arguments"].copy()

                print("\nEdit arguments (press Enter to keep the current value):\n")

                for key, value in new_args.items():
                    new_value = input(f"{key} [{value}]: ").strip()

                    if new_value:
                        # Convert to original type where possible
                        try:
                            if isinstance(value, bool):
                                new_args[key] = new_value.lower() in (
                                    "true",
                                    "1",
                                    "yes",
                                    "y",
                                )
                            elif isinstance(value, int):
                                new_args[key] = int(new_value)
                            elif isinstance(value, float):
                                new_args[key] = float(new_value)
                            else:
                                new_args[key] = new_value
                        except ValueError:
                            print(f"Invalid value for {key}. Keeping original.")

                        decision = {
                            "type": "edit",
                            "edited_action": {
                                "name": request["name"],
                                "args": new_args,
                            },
                        }

            else:
                print("Invalid choice.")
                continue

            stream = agent.stream_events(
                Command(
                    resume={
                        "decisions": [decision]
                    }
                ),
                transformers=[MyCustomTransformer],
                config=config,
                version="v3",
            )

            for name, item in stream.interleave("my_custom", "messages"):
                if name == "my_custom":
                    print(f"Tool update: {item}")
                elif name == "messages":
                    for j in item.text:
                        print(j,end="",flush=True)

        msg = input("\nEnter: ")

In [ ]:
from RAG.agent import search_langchain_docs

In [ ]:
search_agent = create_agent(
    model=model,
    system_prompt="""
        You are an expert LangChain documentation assistant.

        You have access to exactly one tool:
        - search_langchain_docs(query)

        Workflow:
        1. Analyze the user's question.
        2. Create the best search query you can.
        3. Call the search tool. The tool has a limit of 5 calls per run.
        4. If the retrieved documentation is insufficient, ambiguous, or only partially answers the question, reformulate the search query and search again.
        5. Continue until you have enough information.
        6. Base your answer ONLY on the retrieved documentation.
        7. If the documentation does not answer the question, clearly say so instead of guessing.
        8. Cite relevant source paths mentioned by the tool whenever possible.
    """,
    tools=[search_langchain_docs],
    middleware=[ToolCallLimitMiddleware(tool_name=search_langchain_docs.name, run_limit=5)],
    checkpointer=checkpointer,
)

In [ ]:
@tool
def langchain_search_agent(query: str) -> str:
    """Queries a search agent specialized in LangChain documentation.

    Passes a user query to the underlying LangChain agent to retrieve answers, 
    code examples, or API specifications directly from LangChain docs.

    Args:
        query (str): The natural language query or technical question about LangChain.

    Returns:
        str: The final textual response extracted from the agent's message stack.
    """
    result = search_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query,
                }
            ]
        },
    )

    return result["messages"][-1].content

In [ ]:
agent = create_agent(
    model=model,
    system_prompt="You are a NoteBot. each tool you have is designed for processing a single note you need to make multiple tool calls if you want to process more than one note.",
    tools=[*NOTEBOT_TOOLS,langchain_search_agent],
    checkpointer=checkpointer,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                'save_note':True,
                'delete_note':{
                    "allowed_decisions":['approve','reject']
                }
            },
            description_prefix="Tool execution pending approval"
        ),
        ToolRetryMiddleware(
            max_retries=9,
            backoff_factor=1.1
        ),
    ],
    store=store
)

In [ ]:
conn = sqlite3.connect("../db/checkpoint.db",check_same_thread=False)
conn.execute(
    "DELETE FROM checkpoints WHERE thread_id = 'ma0'"
)
conn.commit()

In [ ]:
chat_with_agent(agent,"ma0")

In [ ]:
config={"configurable": {"thread_id": 'ma0'}}
pprint(agent.get_state(config=config).values)